# IMPORTS

In [5]:
from IPython.display import HTML, display
display(HTML('<style>.container { width:90% !important; }</style>'))
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'
import pandas as pd
import openpyxl
from tkinter import simpledialog
import re
import shutil
import os
import requests
import json
from geopy.geocoders import Nominatim
from datetime import timedelta, datetime
from docxtpl import DocxTemplate
import subprocess
import pickle
import os.path
import secrets
from randomtimestamp import randomtimestamp, random_time
from random import randint
from docx import Document
from PyPDF2 import PdfMerger
from IPython.display import FileLink, FileLinks
from time import sleep
from geopy.exc import GeocoderTimedOut
import locale

    pd.get_option('display.max_rows')
    pd.get_option('display.max_columns')
    pd.get_option('display.max_colwidth')
    pd.get_option('display.expand_frame_repr')
    pd.set_option('display.expand_frame_repr', False)

# CONSTANTES

In [6]:
from octobre.october import october

june = [
    "28/06/23", "29/06/23", "30/06/23"
]

july = [
    "03/07/23", "04/07/23", "05/07/23", "06/07/23", "07/07/23",
    "10/07/23", "11/07/23", "12/07/23", "13/07/23",
    "17/07/23", "18/07/23", "19/07/23", "20/07/23", "21/07/23",
    "24/07/23", "25/07/23", "26/07/23", "27/07/23", "28/07/23"
]

sept = [
    "01/09/23",
    "07/09/23", "08/09/23", 
    "11/09/23", "12/09/23", "13/09/23", "14/09/23", "15/09/23",
    "18/09/23", "19/09/23", "20/09/23", "21/09/23", "22/09/23",
    "25/09/23", "26/09/23", "27/09/23", "28/09/23", "29/09/23"
]

octo = [
    "02/10/23", "03/10/23", "04/10/23", "05/10/23", "06/10/23",
    "09/10/23", "10/10/23", "11/10/23", "12/10/23", "13/10/23",
    "16/10/23", "17/10/23", "18/10/23", "19/10/23", "20/10/23",
    "23/10/23", "24/10/23", "25/10/23", "26/10/23", "27/10/23"
]

types = ["itinerary", "call", "park", "poi", "break"]

agenda = {}
for date in june + july + sept + octo:
    agenda[date] = {}

locale.setlocale(locale.LC_ALL, 'fr_FR.UTF-8');
geolocator = Nominatim(user_agent="mySecretAgent")

filepath="/home/julien/website/application/data/xls_x/carnet_de_bord.xlsx"

durations = [0.75, 1, 1.25, 1.5]

zones = pd.read_excel(filepath, sheet_name="zones")
zones.set_index("cp", inplace=True)

rootdir = 'octobre/pdfs/'

cdb = pd.read_excel(filepath, sheet_name="pds")
cdb["prenom"] = cdb["prenom"].fillna("")
cdb["N_P"] = cdb["nom"] + " " + cdb["prenom"].apply(lambda x: x.upper())

for date in october.keys():
    #print(date)
    day = october[date]
    for pds, hour in day.items():
        idx = cdb.loc[cdb["N_P"]==pds].index[0]
        dt = pd.to_datetime(date + " " + hour, dayfirst=True).strftime("%Y-%m-%d %H:%M")
        cdb.loc[idx, "ddv"] = dt
        #print(cdb.loc[idx, "ddv"] )

cdb.to_excel("/home/julien/website/application/data/xls_x/cdb.xlsx", sheet_name="pds")

'fr_FR.UTF-8'

# GEOCODE

In [7]:
def do_geocode(address, attempt=1, max_attempts=5):
    try:
        return geolocator.geocode(address)
    except GeocoderTimedOut:
        if attempt <= max_attempts:
            return do_geocode(address, attempt=attempt+1)
        raise
        

# DOCX TO PDF

In [8]:
def pdf_from_docx(input_docx, output_pathname):

    subprocess.call(
        [
            'soffice',
            # '--headless',
            '--convert-to',
            'pdf',
            '--outdir',
            output_pathname,
            input_docx
        ],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

    return None

# GET DAY

In [9]:
def get_day(date):
    
    us_date = pd.to_datetime(date, dayfirst=True).date().strftime("%Y-%m-%d")
    
    home = {
        "type": "poi",
        "ddv": f"{us_date} 07:30",
        "nom": "HOME",
        "adresse": "35 RUE MARCEL BONTEMPS",
        "cp": 92100,
        "ville": "BOULOGNE BILLANCOURT"  
    }
    lunch = {
        "type": "break",
        "ddv": f"{us_date} 13:30",
        "nom": "LUNCH HOME",
        "adresse": "35 RUE MARCEL BONTEMPS",
        "cp": 92100,
        "ville": "BOULOGNE BILLANCOURT"  
    }
    diner = {
        "type": "poi",
        "ddv": f"{us_date} 19:30",
        "nom": "BACK HOME",
        "adresse": "35 RUE MARCEL BONTEMPS",
        "cp": 92100,
        "ville": "BOULOGNE BILLANCOURT"  
    }

    df_h = pd.DataFrame([home, lunch, diner])
    
    df_h["lat"] = pd.Series(dtype=float, index=df_h.index)
    df_h["lon"] = pd.Series(dtype=float, index=df_h.index)
    
    df_h.columns=[
        "type", "ddv", "id", "adr", "zip", "city", "lat", "lon" 
    ]
    
    for i, row in df_h.iterrows():
        sleep(1)
        loc = do_geocode(f'{row["adr"]} {str(row["zip"])} {row["city"]}')
        sleep(1)
        df_h.loc[i, "lat"] = loc.latitude
        df_h.loc[i, "lon"] = loc.longitude
        
    cdb = pd.read_excel("/home/julien/website/application/data/xls_x/cdb.xlsx", sheet_name="pds")
    cdb["ddv"] = cdb["ddv"].astype(str)
        
    tournee = cdb[["ddv", "nom", "adresse", "cp", "ville"]][cdb["ddv"].str.contains(us_date)].copy()
    tournee.columns = ["ddv", "id", "adr", "zip", "city"]
    #print(tournee)
    
    for i, row in tournee.iterrows():
        
        loc = geolocator.geocode(f'{tournee.loc[i, "adr"]} {str(int(tournee.loc[i, "zip"]))} {tournee.loc[i, "city"]}')
        
        tournee.loc[i, "lat"] = loc.latitude
        tournee.loc[i, "lon"] = loc.longitude
        tournee["type"] = "call"
    
    
    day = pd.concat([tournee, df_h], ignore_index=True)
    day["ddv"] = pd.to_datetime(day["ddv"], format="mixed")
    day = day.sort_values(by=["ddv"]).fillna("")
    day.reset_index(drop=True, inplace=True)
    day["type"] = day["type"].replace("", "call")
    day["zip"] = day["zip"].astype(int)
    
    return day

In [ ]:
day = get_day("02/10/23")
day

# RANDOM DEPARTURE

In [10]:
def get_random_time(date, h_min="08:30", h_max="09:30"):

    t_min = datetime.strptime(f"{h_min}", "%H:%M").time()
    t_max = datetime.strptime(f"{h_max}", "%H:%M").time()
    rand_t = random_time(start=t_min, end=t_max, pattern="%H:%M")
    rand_time = rand_t.strftime("%H:%M")

    return pd.to_datetime(f"{date} {rand_time}", format="%d/%m/%y %H:%M")

# ITINARIES & CAR PARK

    datetime.strptime(date, "%d/%m/%y").date().strftime("%d %B %Y")

In [11]:
def get_itinaries_and_parks(date: str, day: pd.DataFrame):
    
    #print(day)
    
    departure_d_t = get_random_time(date)
    print(f"day begins at {departure_d_t.time()}")
    
    ts = pd.to_datetime(date, dayfirst=True)
    filename = ts.date().strftime("%Y%m%d")
    
    trajets = pd.DataFrame(columns = ["from", "to", "distance", "departure", "duration", "arrival"])
    parkings = pd.DataFrame(columns=["pbp_date", "park", "start", "stay", "stop", "code_zone", "nom_zone", "ville", "price"])
    
    for i in range(day.shape[0]-1):
        
        origin = day.loc[i]
        lat_0 = day.loc[i, "lat"]
        lon_0 = day.loc[i, "lon"]    
        trajets.loc[i, "from"] = origin["id"]

        destination = day.loc[i+1]
        lat_1 = day.loc[i+1, "lat"]
        lon_1 = day.loc[i+1, "lon"]
        trajets.loc[i, "to"] = destination["id"]

        trajets.loc[i, "departure"] = departure_d_t.time()
        # print(f"route departure: {departure_d_t.time()}")
        
        #print(f"http://router.project-osrm.org/route/v1/car/{lon_0},{lat_0};{lon_1},{lat_1}?overview=false""")

        r = requests.get(f"http://router.project-osrm.org/route/v1/car/{lon_0},{lat_0};{lon_1},{lat_1}?overview=false""")
        sleep(1)

        routes = json.loads(r.content)
        route = routes.get("routes")[0]

        duration = int(round(route["duration"]/60,0))
        trajets.loc[i, "duration"] = duration
        # print(f"route duration: {duration} minutes")
        
        arrival_d_t = datetime.combine(departure_d_t.date(), departure_d_t.time()) + timedelta(minutes = duration)
        trajets.loc[i, "arrival"] = arrival_d_t.time()
        # print(f"route end: {arrival_d_t.time()}")
        
        distance = round(route['distance']/1000, 1)    
        trajets.loc[i, "distance"] = distance
        
        
        if destination["type"] == "call":            
            
            start = arrival_d_t + timedelta(minutes=9)
            # print(f"parking start: {start.time()}")
            
            parkings.loc[i, "pbp_date"] = datetime.strptime(date, "%d/%m/%y").date().strftime("%d %B %Y")
            parkings.loc[i, "park"] = f'PARK_{destination["id"]}'            
            parkings.loc[i, "start"] = start.time().strftime("%H:%M")
            parkings.loc[i, "stay"] = durations[randint(0,3)]
            # print(f'parking duration: {parkings.loc[i, "stay"]*60} minutes')
            
            stop = start + timedelta(minutes=parkings.loc[i, "stay"]*60)
            # print(f"parking end: {stop.time()}")
            
            parkings.loc[i, "stop"] = stop.time().strftime("%H:%M")
            code_zone = destination["zip"]
            parkings.loc[i, "code_zone"] = code_zone
            parkings.loc[i, "nom_zone"] = zones.loc[code_zone, "nom zone"]
            parkings.loc[i, "pbp_ville"] = zones.loc[code_zone, "ville"]
            price = (parkings.loc[i, "stay"] * zones.loc[code_zone, "tarif"]) + 0.25
            parkings.loc[i, "price"] = price
            
            departure_d_t = stop + timedelta(minutes=8)
            
            
        elif destination["type"] == "break":
            
            start = arrival_d_t + timedelta(minutes=9)
            # print(f"break start: {start.time()}")
            
            break_duration = 45
            # print(f"break duration: {break_duration} minutes")
            
            stop = start + timedelta(minutes=break_duration)
            # print(f"break stop: {stop.time()}")
            
            departure_d_t = stop + timedelta(minutes=8)
            
        
        # elif destination["type"] == "poi":
            
            # print(f"end of the day: {arrival_d_t.time()}")
            
            
    return trajets, parkings

In [ ]:
t, p = get_itinaries_and_parks(date="02/10/23", day=day)
p

# GET INVOICES

In [12]:
def get_pbp_invoices(date: str):
    
    day = get_day(date)      
    trajets, parkings = get_itinaries_and_parks(date, day)
    parkings.reset_index(drop=True, inplace=True)
    parkings["price"] = parkings["price"].replace(".", ",", regex=False)
    ts = pd.to_datetime(date, dayfirst=True)
    filename = ts.date().strftime("%Y%m%d")
    
    
    with pd.ExcelWriter(f"/home/julien/website/frais/tournées/{filename}.xlsx") as writer:
        
        day.to_excel(writer, sheet_name="day", index=False)
        trajets.to_excel(writer, sheet_name="trajets", index=False)
        parkings.to_excel(writer, sheet_name="parkings", index=False)
    
    prix_total = "{:.2f}".format(parkings["price"].sum())
    dist_total = int(round(trajets["distance"].sum(), 0))    
    month = ts.date().strftime("%B")
    
    common_path = f"{month}"
    
    for j, parking in parkings.iterrows():
        
        park = {
            "date_2": date,
            "date": parking["pbp_date"],
            "start": parking["start"],
            "end": parking["stop"],
            "ville": parking["pbp_ville"],
            "nom": parking["nom_zone"],
            "code": parking["code_zone"],
            "price": "{:.2f}".format(parking["price"]).replace(".", ",")
        }
        
        page = DocxTemplate("/home/julien/website/frais/pbp_tpl.docx")
        page.render(park)
        path_docx = f"{common_path}/docx/pbp_{filename}_{j+1}.docx"
        page.save(path_docx)
        pdf_from_docx(path_docx, f"{common_path}/pdfs/")
        os.remove(path_docx)
                
    return prix_total, dist_total, j+1

  # EXECUTE

In [13]:
execute = {}

In [14]:
def process_date(date):
    
    print(date)
    
    prix_total, dist_total, n = get_pbp_invoices(date)
    
    print(f"n_kms: \t\t{dist_total}\nparks:\t\t{prix_total}\nn_pdfs:\t\t{n}\n")

    date = datetime.strptime(date, "%d/%m/%y").date().strftime("%Y%m%d")
    
    execute[date] = {
        "total parkings" : prix_total,
        "total kms": dist_total,
        "filepaths": [f"octobre/pdfs/pbp_{date}_{x+1}.pdf" for x in range(n)]
    }
    
    return execute

def make_clickable(url_list):
    
    names = [os.path.basename(url) for url in url_list]    
    
    return ['<a href="{}">{}</a>'.format(url, name) for url, name in zip(url_list, names)]

    for date in june:
        execute = process_date(date)

    for date in july:
        execute = process_date(date)

In [15]:
for date in octo:
    execute = process_date(date)

02/10/23
day begins at 09:18:00
n_kms: 		22
parks:		21.00
n_pdfs:		4

03/10/23
day begins at 08:37:00
n_kms: 		19
parks:		14.75
n_pdfs:		3

04/10/23
day begins at 09:20:00
n_kms: 		37
parks:		12.75
n_pdfs:		3

05/10/23
day begins at 08:35:00
n_kms: 		26
parks:		21.00
n_pdfs:		4

06/10/23
day begins at 09:26:00
n_kms: 		26
parks:		21.00
n_pdfs:		4

09/10/23
day begins at 09:15:00
n_kms: 		31
parks:		28.00
n_pdfs:		4

10/10/23
day begins at 09:19:00
n_kms: 		39
parks:		31.75
n_pdfs:		5

11/10/23
day begins at 08:34:00
n_kms: 		39
parks:		26.25
n_pdfs:		5

12/10/23
day begins at 08:53:00
n_kms: 		14
parks:		7.50
n_pdfs:		2

13/10/23
day begins at 09:28:00
n_kms: 		33
parks:		12.75
n_pdfs:		3

16/10/23
day begins at 08:41:00
n_kms: 		40
parks:		17.00
n_pdfs:		4

17/10/23
day begins at 09:01:00
n_kms: 		40
parks:		22.50
n_pdfs:		4

18/10/23
day begins at 09:05:00
n_kms: 		24
parks:		9.50
n_pdfs:		2

19/10/23
day begins at 08:50:00
n_kms: 		59
parks:		20.00
n_pdfs:		4

20/10/23
day begins at

In [ ]:
resume = pd.DataFrame.from_dict(execute, orient="index")

In [ ]:
resume.index = pd.to_datetime(resume.index, format="%Y%m%d").strftime('%d/%m/%y')

In [ ]:
resume.style.format({'filepath': make_clickable})

# UPDATE XLSX BOOK

In [ ]:
def update_calls_sheet(date, filepath=filepath):
    
    calls_of_the_day = get_calls_by_date(date)
    print(f"nb of calls: {calls_of_the_day.shape[0]-3}")
        
    with pd.ExcelWriter(
        filepath, engine='openpyxl', if_sheet_exists="replace", mode="a", date_format="dd/mm/yy", datetime_format="HH:MM"
    ) as writer:

        if "calls" not in writer.sheets:
            
            calls_of_the_day.to_excel(writer, sheet_name="calls", index=False)  
            print("Creating sheet 'calls'")
            # print("Calls: sheet updated")
            
        else:
            
            existing_calls = pd.read_excel(filepath, sheet_name="calls", engine='openpyxl')

            dates = set([row["ddv"].date() for i, row in existing_calls.iterrows()])
            
            ts = pd.to_datetime(date, format="%d/%m/%y")

            if ts.date() in dates:

                print("date already in calls")
                # print("no need to update sheet calls")

            else:

                concat = pd.concat([existing_calls, calls_of_the_day])
                
                concat.to_excel(writer, sheet_name="calls", index=False)
                
                # print("Calls: sheet updated")

    return None 

In [ ]:
def update_kms_sheet(day_steps, filepath=filepath):

    date = day_steps[0]
    dist_tot_park_tot_dict = {}
    dist_tot_park_tot_dict[date] = {}
    dist_tot_park_tot_dict[date]["kms"] = day_steps[-2]
    dist_tot_park_tot_dict[date]["parkings"] = day_steps[-1]
    
    kms_of_the_day = pd.DataFrame.from_dict(dist_tot_park_tot_dict, orient="index")
    kms_of_the_day.reset_index(drop=False, inplace=True)
    kms_of_the_day.columns=["date", "kms pro", "total parking"]
            
    with pd.ExcelWriter(
        filepath, engine='openpyxl', if_sheet_exists="replace", mode="a", date_format="dd/mm/yy", datetime_format="HH:MM"
    ) as writer:    
        
        if "kms" not in writer.sheets:
            
            kms_of_the_day.to_excel(writer, sheet_name="kms", index=False)  
            print("Creating sheet 'kms'")
            # print("Kms: sheet updated")
            
        else:

            kmsdf = pd.read_excel(filepath, sheet_name="kms", engine='openpyxl')

            dates = set([row["date"] for i, row in kmsdf.iterrows()])

            if date in dates:

                print("date already in kms")
                # print("no need to update sheet kms")

            else:
                
                concat = pd.concat([kmsdf, kms_of_the_day])
                concat.to_excel(writer, sheet_name="kms", index=False)

                # print("Kilometers: sheet updated")
                                
    return None

In [ ]:
day_steps = get_day_steps("28/06/23")

In [ ]:
day_steps[-1]

In [ ]:
def update_sheets(date, filepath=filepath):

    update_calls_sheet(date)
    update_kms_sheet(date)

    return None